# 11 · Next step (Markov chain)

Given what an attacker has done so far, what will they do next? The proposal's model: a first-order **Markov chain**, P(next step | current step), counted from attacker step sequences (`ml/next_step.py`).

Data: **camp7** (development), seed 7, both sites, 60 *campaigns* per site: attackers that go through 3 to 5 steps in a row from one IP (scan, enum, bruteforce, sqli, xss, traversal, cmdi). The generator picks each next step from a probability table we wrote (`CAMPAIGN_NEXT` in `lab/generator/generate_traffic.py`).

**Honest note:** the chain learns that table, not real attackers. Because the next step is random, even the true table cannot be right every time; section 3 shows that ceiling.

In [1]:
import itertools, re, ast, sys
from collections import Counter
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from eval.final_test import run
from ml.next_step import STEPS, NextStepModel, merge_steps, session_steps

D = ROOT / 'data'
SITES = {'Juice Shop': ('juiceshop_generated_camp7.log', 'requests_camp7.csv', 'sessions_camp7.csv'),
         'DVWA': ('dvwa_generated_camp7.log', 'requests_dvwa_camp7.csv', 'sessions_dvwa_camp7.csv')}
data = {}
for site, (log, labels, sess) in SITES.items():
    req, sessions, report = run(D / 'raw' / log, D / 'labels' / labels)
    lab = pd.read_csv(D / 'labels' / sess, keep_default_na=False)
    camp = lab[lab['attack_type'] == 'campaign'].set_index('ip')['steps']
    found = session_steps(req)
    ip_of = req.drop_duplicates('session').set_index('session')['ip']
    detected = {ip_of[s]: st for s, st in found.items()}
    truth = {ip: merge_steps(s.split('>'), min_run=1) for ip, s in camp.items()}
    data[site] = {'truth': truth, 'detected': detected, 'req': req, 'sessions': sessions}
    print(site, report)

Juice Shop {'log_lines': 6372, 'labels': 6372, 'matched': 6372, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


DVWA {'log_lines': 6456, 'labels': 6387, 'matched': 6387, 'unmatched_log_lines': 69, 'unmatched_labels': 0}


## 1. Does the system read the steps right?

For each campaign: the step order the system reads from the log (detector + session layer + attack type) against the order the generator really used. Normal users should have no steps at all.

In [2]:
rows = []
for site, d in data.items():
    t, det = d['truth'], d['detected']
    exact = sum(det.get(ip) == s for ip, s in t.items())
    normal_ips = set(d['req'].loc[d['req']['attack_type'] == '', 'ip']) - set(d['req'].loc[d['req']['attack_type'] != '', 'ip'])
    rows.append({'site': site, 'campaigns': len(t), 'order read exactly': exact,
                 'normal users with steps': sum(ip in det for ip in normal_ips)})
display(pd.DataFrame(rows).set_index('site'))
for site, d in data.items():
    wrong = [(ip, '>'.join(s), '>'.join(d['detected'].get(ip, []))) for ip, s in d['truth'].items() if d['detected'].get(ip) != s]
    print(site, 'examples read differently (true / read):')
    for w in wrong[:5]: print('  ', w)

,campaigns,order read exactly,normal users with steps
site,,,
Juice Shop,60,57,0
DVWA,60,38,0


Juice Shop examples read differently (true / read):
   ('10.1.0.33', 'bruteforce>xss>cmdi>traversal>cmdi', 'xss>cmdi>traversal>cmdi')
   ('10.1.0.122', 'enum>bruteforce>enum', 'bruteforce>enum')
   ('10.1.0.242', 'scan>enum>bruteforce>enum>sqli', 'scan>enum>bruteforce>scan>sqli')
DVWA examples read differently (true / read):
   ('10.1.0.8', 'enum>cmdi>traversal', 'bruteforce>enum>cmdi>traversal')
   ('10.1.0.10', 'scan>sqli>cmdi>traversal', 'scan>sqli>cmdi>bruteforce>traversal')
   ('10.1.0.18', 'scan>sqli>enum>bruteforce>sqli', 'scan>sqli>bruteforce>enum>bruteforce>sqli')
   ('10.1.0.19', 'scan>traversal>sqli>cmdi>xss', 'scan>traversal>bruteforce>sqli>cmdi>xss')
   ('10.1.0.44', 'bruteforce>traversal>sqli>traversal>cmdi', 'bruteforce>traversal>sqli>traversal>bruteforce>cmdi')


## 2. Next-step accuracy on the other site

Trained on the true step orders of one site, tested on the steps the system **read** from the other site's log (what it would see in practice). *top-k*: the real next step is among the k most likely ones.

In [3]:
def baseline_most_common(train, test):
    top = Counter(b for s in train for b in s[1:]).most_common(1)[0][0]
    pairs = [(a, b) for s in test for a, b in zip(s, s[1:])]
    return sum(b == top for _, b in pairs) / len(pairs)

rows = []
for train_site, test_site in [('Juice Shop', 'DVWA'), ('DVWA', 'Juice Shop')]:
    train = list(data[train_site]['truth'].values())
    test = list(data[test_site]['detected'].values())
    model = NextStepModel.fit(train)
    row = {'train': train_site, 'test': test_site}
    for k in (1, 2, 3):
        acc, n = model.top_k_accuracy(test, k)
        row[f'top-{k}'] = round(acc, 3)
    row['transitions tested'] = n
    row['always the most common step'] = round(baseline_most_common(train, test), 3)
    row['random guess'] = round(1 / (len(STEPS) - 1), 3)
    rows.append(row)
pd.DataFrame(rows).set_index(['train', 'test'])

,,top-1,top-2,top-3,transitions tested,always the most common step,random guess
train,test,,,,,,
Juice Shop,DVWA,0.321,0.584,0.746,209,0.254,0.167
DVWA,Juice Shop,0.398,0.674,0.873,181,0.287,0.167


## 3. The ceiling: how often can *any* model be right?

The generator picks the next step at random from `CAMPAIGN_NEXT`. Even a model that knows that table exactly can only guess its most likely entry, so its top-1 accuracy is the best possible.

In [4]:
src = (ROOT / 'lab' / 'generator' / 'generate_traffic.py').read_text()
NEXT = ast.literal_eval(re.search(r'CAMPAIGN_NEXT = (\{.*?\n\})', src, re.S).group(1))
for site, d in data.items():
    pairs = [(a, b) for s in d['truth'].values() for a, b in zip(s, s[1:])]
    best = sum(b == max(NEXT[a], key=NEXT[a].get) for a, b in pairs) / len(pairs)
    print(f'{site}: top-1 of the true table = {best:.3f}')

Juice Shop: top-1 of the true table = 0.377
DVWA: top-1 of the true table = 0.377


## 4. The chain the system uses

Trained on both sites' campaigns (`python -m ml.train_next_step`). Rows: current step; columns: next step.

In [5]:
model = NextStepModel.fit(list(data['Juice Shop']['truth'].values()) + list(data['DVWA']['truth'].values()))
display(model.transitions.round(2))
print('after a scan:', model.predict('scan'))

,scan,enum,bruteforce,sqli,xss,traversal,cmdi
scan,0.00,0.17,0.24,0.19,0.06,0.27,0.06
enum,0.02,0.00,0.26,0.41,0.17,0.08,0.08
bruteforce,0.02,0.33,0.00,0.29,0.09,0.19,0.09
sqli,0.01,0.27,0.01,0.00,0.39,0.16,0.16
xss,0.02,0.02,0.02,0.60,0.00,0.15,0.19
traversal,0.02,0.02,0.02,0.30,0.30,0.00,0.34
cmdi,0.03,0.03,0.03,0.18,0.34,0.39,0.00


after a scan: [('traversal', 0.269), ('bruteforce', 0.244), ('sqli', 0.192)]


## 5. Conclusions

- The system reads the step order of most campaigns from the log, and no normal user gets any step.
- Next-step top-1 accuracy is far above a random guess and above always guessing the most common step, and it is close to the ceiling of section 3: the chain learns the scenario table about as well as is possible.
- Top-2 / top-3 are the useful numbers for an investigator: the real next step is usually among the first two or three shown.
- **Limitations:** the chain learns our scenarios, not real attackers. On DVWA each campaign step logs in again (a generator habit), which the session layer sometimes reads as extra brute force, so some DVWA orders are read with an extra step.